# Etapa 0: análise exploratória qualitativa e quantitativa

Antes do ETL, este notebook examina as fontes de dados do Radar Epidemiológico para
responder às perguntas que definem o pipeline e o modelo:

1. A API DEMAS exige autenticação? Como pagina de fato? Quais filtros aceita?
2. Qual é o esquema de cada endpoint, e onde ele é inconsistente (tipos, ausências, códigos)?
3. Qual é a qualidade do cadastro de UBS (coordenadas, CNES, cobertura municipal)?
4. Qual é o volume de notificações e quanto custa extraí-lo pela API?
5. Como se comporta a série semanal: sazonalidade, sobredispersão, anos epidêmicos?
6. Que decisões de ETL e de modelagem decorrem disso?

**Restrição do projeto:** nenhum dado é gravado em disco. Tudo é baixado para a memória e
descartado ao fechar o kernel. Antes de versionar o notebook, limpe as saídas
(*Edit > Clear Outputs of All Cells*), para que os dados não entrem no repositório.

Execução: `uv run jupyter lab notebooks/`. Rodar o notebook inteiro leva poucos minutos e
faz algumas centenas de requisições à DEMAS.

In [ ]:
import asyncio
import math
import time

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

from etl.extract.demas import ENDPOINTS, DemasClient
from etl.extract.ibge import municipios as carregar_municipios
from etl.transform.calendario import calendario
from etl.transform.normalize import (
    ausente,
    codigo_cnes,
    codigo_ibge6,
    coordenada,
    dentro_do_brasil,
)
from ml.evaluate import baseline_sazonal, mae, mase

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)

# Amostra temporal. O filtro id_municip da API usa o município da unidade notificadora.
MUNICIPIO_AMOSTRA = "260960"  # Olinda (PE): de 1 a 2 mil notificações de dengue por ano
ANOS_AMOSTRA = range(2015, 2026)  # zika começa em 2015; 2026 ainda está em curso
DOENCAS_SINAN = ["dengue", "chikungunya", "zika"]

cliente = DemasClient()

## Parte 1: análise qualitativa

### 1.1 Inventário das fontes

| Fonte | Conteúdo | Granularidade | Chave de junção | Uso no projeto |
|---|---|---|---|---|
| DEMAS `/arboviroses/dengue` | Notificações do SINAN | Uma linha por notificação | `id_municip`, `id_mn_resi` (IBGE 6) | Dashboard, previsão |
| DEMAS `/arboviroses/chikungunya` | Notificações do SINAN | Uma linha por notificação | `id_municip`, `id_mn_resi` (IBGE 6) | Dashboard, previsão |
| DEMAS `/arboviroses/zikavirus` | Notificações do SINAN | Uma linha por notificação | `id_municip`, `id_mn_resi` (IBGE 6) | Dashboard, previsão |
| DEMAS `/arboviroses/febre-amarela-humanos-primatas-nao-humanos` | Casos humanos de febre amarela | Uma linha por caso | `cod_mun_lpi` (IBGE 6, local provável de infecção) | Dashboard |
| DEMAS `/assistencia-a-saude/unidade-basicas-de-saude` | Cadastro de UBS com coordenadas | Uma linha por unidade | `cnes`, `ibge` (IBGE 6) | Mapa |
| DEMAS `/cnes/estabelecimentos/{cnes}` | Detalhe do estabelecimento | Um estabelecimento | `codigo_cnes` | Popup do mapa (tempo real) |
| IBGE Localidades | Municípios | Um município | código de 7 dígitos | Dicionário código ↔ nome |
| BrasilAPI / AwesomeAPI | CEP para lat/lon | Um CEP | — | Mapa (tempo real) |
| Manuais do MS | PDFs de manejo clínico e vigilância | Página | manual + página | Corpus do RAG |

As fontes do SINAN e a de febre amarela têm esquemas diferentes: a febre amarela usa
local provável de infecção, datas `dd/mm/aaaa` e semana epidemiológica numérica, e não
traz classificação final nem município de notificação.

### 1.2 Conformidade da API com a documentação

O AGENTS.md e o Swagger fazem afirmações sobre autenticação e paginação. Cada uma é
testada abaixo contra a API real.

O parâmetro `campos`, que restringe as colunas devolvidas, não é testado aqui de propósito:
em 2026-10-28 ele levou a consulta de 1 s para 36 s (limit=10) e para HTTP 502 por timeout
do gateway (limit=100 ou mais). As colunas são selecionadas depois do download.

In [ ]:
async def verificar_api() -> pd.DataFrame:
    ubs, dengue = ENDPOINTS["ubs"].caminho, ENDPOINTS["dengue"].caminho
    grande = await cliente.pagina(ubs, limit=5000)
    desloc0 = await cliente.pagina(ubs, offset=0, limit=3)
    desloc1 = await cliente.pagina(ubs, offset=1, limit=3)
    filtro6 = await cliente.pagina(dengue, limit=1, nu_ano="2024", id_municip="355030")
    filtro7 = await cliente.pagina(dengue, limit=1, nu_ano="2024", id_municip="3550308")
    por_registro = desloc1[0] == desloc0[1]
    return pd.DataFrame(
        [
            ("Autenticação", "Vários endpoints marcados como autenticados", "HTTP 200 sem token"),
            ("Tamanho máximo da página", "AGENTS.md presumia 20", f"{len(grande)} com limit=5000"),
            (
                "Semântica do offset",
                "Swagger: número da página",
                "deslocamento por registro" if por_registro else "número da página",
            ),
            ("id_municip com 6 dígitos", "não documentado", f"{len(filtro6)} registro(s)"),
            ("id_municip com 7 dígitos", "não documentado", f"{len(filtro7)} registro(s)"),
        ],
        columns=["item", "documentado", "observado"],
    ).set_index("item")


verificacao_api = await verificar_api()
verificacao_api

### 1.3 Esquema e perfil das colunas

Uma página de 1000 registros por endpoint. A página não é uma amostra aleatória (a API
devolve os registros em ordem de armazenamento), então serve para conhecer o esquema e
os formatos, não para estimar proporções.

In [ ]:
def perfil(registros: list[dict]) -> pd.DataFrame:
    """Tipos observados, percentual ausente (inclusive a string "nan") e exemplos."""
    df = pd.DataFrame(registros)
    linhas = []
    for coluna in df.columns:
        valores = df[coluna]
        presentes = valores[~valores.map(ausente)]
        linhas.append(
            {
                "coluna": coluna,
                "tipos": ", ".join(sorted({type(v).__name__ for v in valores})),
                "pct_ausente": round(100 * (1 - len(presentes) / len(valores)), 1),
                "pct_string_nan": round(100 * (valores == "nan").mean(), 1),
                "distintos": presentes.astype(str).nunique(),
                "exemplo": presentes.iloc[0] if len(presentes) else None,
            }
        )
    return pd.DataFrame(linhas).set_index("coluna")


nomes = ["dengue", "chikungunya", "zika", "febre_amarela", "ubs"]
paginas = await asyncio.gather(
    cliente.pagina(ENDPOINTS["dengue"].caminho, nu_ano="2024"),
    cliente.pagina(ENDPOINTS["chikungunya"].caminho, nu_ano="2024"),
    cliente.pagina(ENDPOINTS["zika"].caminho, nu_ano="2024"),
    cliente.pagina(ENDPOINTS["febre_amarela"].caminho),
    cliente.pagina(ENDPOINTS["ubs"].caminho),
)
amostras_esquema = dict(zip(nomes, paginas, strict=True))
perfis = {nome: perfil(registros) for nome, registros in amostras_esquema.items()}

pd.DataFrame(
    {
        nome: {
            "registros": len(p),
            "colunas": len(perfis[nome]),
            "colunas 100% ausentes": int((perfis[nome]["pct_ausente"] == 100).sum()),
            "colunas com tipo misto": int(perfis[nome]["tipos"].str.contains(",").sum()),
        }
        for nome, p in amostras_esquema.items()
    }
).T

In [ ]:
for nome in ["ubs", "febre_amarela"]:
    display(Markdown(f"#### {nome}"))
    display(perfis[nome])

In [ ]:
# Os três agravos do SINAN: perfil lado a lado das colunas comuns.
colunas = {d: set(perfis[d].index) for d in DOENCAS_SINAN}
comuns = sorted(set.intersection(*colunas.values()))
print(f"{len(comuns)} colunas comuns aos três agravos")
for d in DOENCAS_SINAN:
    outras = set.union(*(c for k, c in colunas.items() if k != d))
    print(f"{d}: {len(colunas[d])} colunas, {len(colunas[d] - outras)} exclusivas")

lado_a_lado = pd.concat(
    {d: perfis[d].loc[comuns, ["tipos", "pct_ausente", "exemplo"]] for d in DOENCAS_SINAN},
    axis=1,
)
lado_a_lado

In [ ]:
# Colunas comuns cujo tipo Python muda entre agravos ou dentro de um mesmo agravo.
tipos = pd.DataFrame({d: perfis[d].loc[comuns, "tipos"] for d in DOENCAS_SINAN})
tipos[(tipos.nunique(axis=1) > 1) | tipos.apply(lambda c: c.str.contains(",")).any(axis=1)]

### 1.4 Semântica dos campos-chave e decisões que dependem dela

Códigos conforme o dicionário de dados do SINAN (a confirmar na versão oficial de cada
agravo antes de fixar no ETL):

| Campo | Significado | Observação |
|---|---|---|
| `id_municip` | Município da unidade notificadora | É o campo do filtro da API |
| `id_mn_resi` | Município de residência | Padrão dos boletins do MS para incidência |
| `dt_sin_pri`, `sem_pri` | Data e semana epidemiológica de início dos sintomas | Os boletins agregam por esta semana |
| `dt_notific`, `sem_not` | Data e semana da notificação | Atrasada em relação ao início dos sintomas |
| `nu_ano` | Ano da notificação | É o campo do filtro da API |
| `classi_fin` | Classificação final | Dengue/chikungunya: 5 descartado, 8 inconclusivo, 10 dengue, 11 com sinais de alarme, 12 grave, 13 chikungunya |
| `criterio` | Critério de confirmação | 1 laboratorial, 2 clínico-epidemiológico, 3 em investigação |
| `evolucao` | Evolução do caso | 1 cura, 2 óbito pelo agravo, 3 óbito por outras causas, 4 óbito em investigação, 9 ignorado |
| `nu_idade_n` | Idade codificada | Primeiro dígito é a unidade (4 = anos), os demais o valor |

Decisões que a parte quantitativa ajuda a tomar:

- **Município de referência:** residência (`id_mn_resi`) mede onde a doença ocorre;
  notificação (`id_municip`) mede a carga sobre a rede. A API só filtra pela notificação.
- **Semana de referência:** início dos sintomas (`sem_pri`) segue o padrão epidemiológico,
  mas as semanas recentes ficam incompletas por causa do atraso de notificação.
- **Definição de caso:** notificações totais ou casos prováveis (notificações menos
  descartados).

## Parte 2: análise quantitativa

### 2.1 Municípios do IBGE

In [ ]:
municipios = carregar_municipios()
print(f"{len(municipios)} municípios")
print(f"código de 6 dígitos é único: {municipios['codigo_ibge6'].is_unique}")
municipios.groupby("regiao").size().rename("municípios").to_frame().T

### 2.2 Cadastro de UBS

Download completo, para medir o custo real e a qualidade do cadastro inteiro.

In [ ]:
inicio = time.perf_counter()
ubs_bruto = pd.DataFrame(await cliente.todos(ENDPOINTS["ubs"].caminho))
tempo_ubs = time.perf_counter() - inicio

ubs = ubs_bruto.assign(
    latitude=ubs_bruto["latitude"].map(coordenada),
    longitude=ubs_bruto["longitude"].map(coordenada),
    cnes=ubs_bruto["cnes"].map(codigo_cnes),
    codigo_ibge6=ubs_bruto["ibge"].map(codigo_ibge6),
)
ubs["coordenada_valida"] = [
    dentro_do_brasil(la, lo) for la, lo in zip(ubs["latitude"], ubs["longitude"], strict=True)
]
sem_coordenada = ubs["latitude"].isna() | ubs["longitude"].isna()
casas_decimais = ubs_bruto["latitude"].astype(str).str.split(",").str[1].str.len()

qualidade_ubs = pd.Series(
    {
        "registros": len(ubs),
        "requisições": math.ceil(len(ubs) / 1000),
        "tempo de download (s)": round(tempo_ubs, 1),
        "coordenada ausente ou ilegível": int(sem_coordenada.sum()),
        "coordenada fora do Brasil": int((~ubs["coordenada_valida"] & ~sem_coordenada).sum()),
        "latitude com até 2 casas decimais (~1 km)": int((casas_decimais <= 2).sum()),
        "CNES inválido": int(ubs["cnes"].isna().sum()),
        "linhas com CNES repetido": int(ubs["cnes"].duplicated(keep=False).sum()),
        "linhas idênticas repetidas": int(ubs_bruto.duplicated().sum()),
        "código IBGE fora do dicionário": int(
            (~ubs["codigo_ibge6"].isin(municipios["codigo_ibge6"])).sum()
        ),
        "UF diferente do prefixo do código IBGE": int(
            (ubs["uf"] != ubs["codigo_ibge6"].str[:2]).sum()
        ),
        "municípios sem nenhuma UBS": int(
            (~municipios["codigo_ibge6"].isin(ubs["codigo_ibge6"])).sum()
        ),
    }
)
qualidade_ubs.to_frame("valor")

In [ ]:
# CNES repetidos: mesma unidade com dados diferentes, ou cópia exata?
repetidos = ubs[ubs["cnes"].duplicated(keep=False)].sort_values("cnes")
print(f"{repetidos['cnes'].nunique()} CNES aparecem mais de uma vez")
colunas_que_variam = (
    repetidos.groupby("cnes")[["nome", "latitude", "longitude", "logradouro", "codigo_ibge6"]]
    .nunique()
    .gt(1)
    .mean()
    .mul(100)
    .round(1)
)
display(colunas_que_variam.rename("% dos CNES repetidos em que a coluna varia").to_frame())
repetidos.head(10)

In [ ]:
ubs_validas = ubs[ubs["coordenada_valida"]]
por_uf = (
    ubs_validas.merge(
        municipios[["codigo_ibge6", "uf_sigla"]].drop_duplicates(), on="codigo_ibge6", how="left"
    )
    .groupby("uf_sigla")
    .size()
    .sort_values()
)
px.bar(
    por_uf,
    orientation="h",
    title="UBS com coordenada válida, por UF",
    labels={"value": "UBS", "uf_sigla": "UF"},
    height=650,
).update_layout(showlegend=False)

In [ ]:
px.scatter_map(
    ubs_validas.sample(min(8000, len(ubs_validas)), random_state=0),
    lat="latitude",
    lon="longitude",
    hover_name="nome",
    zoom=3,
    height=600,
    map_style="open-street-map",
    title="Amostra de 8 mil UBS",
).update_traces(marker={"size": 3})

### 2.3 Volume de notificações e custo de extração

Download de todas as notificações do município da amostra, por agravo e ano. Só as
colunas usadas na análise são mantidas em memória. O tempo e o número de requisições
medidos aqui servem para extrapolar o custo do ETL.

In [ ]:
CAMPOS_SINAN = [
    "nu_ano", "sem_not", "sem_pri", "dt_notific", "dt_sin_pri", "id_municip", "id_mn_resi",
    "classi_fin", "criterio", "evolucao", "cs_sexo", "nu_idade_n", "tp_sistema",
]  # fmt: skip


async def baixar(doenca: str, ano: int) -> tuple[str, int, list[dict], float]:
    inicio = time.perf_counter()
    registros = await cliente.todos(
        ENDPOINTS[doenca].caminho, nu_ano=str(ano), id_municip=MUNICIPIO_AMOSTRA
    )
    enxutos = [{c: r.get(c) for c in CAMPOS_SINAN} for r in registros]
    return doenca, ano, enxutos, time.perf_counter() - inicio


inicio = time.perf_counter()
baixados = await asyncio.gather(*(baixar(d, a) for d in DOENCAS_SINAN for a in ANOS_AMOSTRA))
tempo_total = time.perf_counter() - inicio

custo = pd.DataFrame(
    [
        {
            "doenca": d,
            "ano": a,
            "registros": len(r),
            "requisicoes": len(r) // 1000 + 1,
            "segundos": s,
        }
        for d, a, r, s in baixados
    ]
)
notificacoes = pd.concat(
    [pd.DataFrame(r).assign(doenca=d) for d, _, r, _ in baixados if r], ignore_index=True
)
print(
    f"{len(notificacoes)} notificações, {custo['requisicoes'].sum()} requisições, "
    f"{tempo_total:.0f} s de relógio"
)
custo.pivot(index="ano", columns="doenca", values="registros")

Para extrapolar o custo nacional: em sondagens de 2026-10-28, uma página de 1000
registros (cerca de 3,7 MB) levou de 1 a 2 segundos com filtro de município e ano, e
de 15 a 45 segundos com `offset` alto sem filtro de município (dengue, 2024). A dengue de
2024 tem mais de 5 milhões de notificações: mais de 5 mil requisições e cerca de 19 GB
de JSON só nesse ano.

### 2.4 Qualidade das notificações da amostra

In [ ]:
amostra = notificacoes.copy()
for coluna in ["dt_notific", "dt_sin_pri"]:
    amostra[coluna] = pd.to_datetime(
        amostra[coluna].where(~amostra[coluna].map(ausente)), errors="coerce"
    )
for coluna in ["id_municip", "id_mn_resi"]:
    amostra[coluna] = amostra[coluna].map(codigo_ibge6)
amostra["classi_fin"] = amostra["classi_fin"].astype(str).str.replace(".0", "", regex=False)
amostra["atraso_dias"] = (amostra["dt_notific"] - amostra["dt_sin_pri"]).dt.days

ausencias = (
    amostra.groupby("doenca")[CAMPOS_SINAN].agg(lambda c: round(100 * c.map(ausente).mean(), 1)).T
)
ausencias.columns.name = "% ausente"
ausencias

In [ ]:
def indicadores(g: pd.DataFrame) -> pd.Series:
    ano_sem_pri = pd.to_numeric(g["sem_pri"].astype(str).str[:4], errors="coerce")
    return pd.Series(
        {
            "notificações": len(g),
            "% residentes em outro município": round(
                100 * (g["id_mn_resi"] != g["id_municip"]).mean(), 1
            ),
            "% ano de sem_pri ≠ nu_ano": round(
                100 * (ano_sem_pri != pd.to_numeric(g["nu_ano"])).mean(), 1
            ),
            "atraso mediano (dias)": g["atraso_dias"].median(),
            "atraso p90 (dias)": g["atraso_dias"].quantile(0.9),
            "% atraso negativo": round(100 * (g["atraso_dias"] < 0).mean(), 2),
            "% atraso > 30 dias": round(100 * (g["atraso_dias"] > 30).mean(), 1),
        }
    )


amostra.groupby("doenca")[amostra.columns].apply(indicadores).T

In [ ]:
# Classificação final: proporção de cada código, por agravo.
(
    amostra.groupby("doenca")["classi_fin"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .unstack(0)
    .fillna(0)
)

In [ ]:
# Os códigos de classi_fin mudam conforme o sistema de origem do registro (tp_sistema)?
amostra["tp_sistema"] = amostra["tp_sistema"].astype(str).str.replace(".0", "", regex=False)
pd.crosstab([amostra["doenca"], amostra["tp_sistema"]], amostra["classi_fin"], margins=True)

In [ ]:
# Sistema de origem por ano: em que ano cada agravo migrou de sistema?
pd.crosstab(amostra["nu_ano"], [amostra["doenca"], amostra["tp_sistema"]])

### 2.5 Série semanal: sazonalidade, sobredispersão e anos epidêmicos

Contagem de notificações por semana epidemiológica de início dos sintomas, com as
semanas sem notificação completadas com zero pelo calendário epidemiológico.

In [ ]:
amostra["ano_epi"] = pd.to_numeric(amostra["sem_pri"].astype(str).str[:4], errors="coerce")
amostra["semana_epi"] = pd.to_numeric(amostra["sem_pri"].astype(str).str[4:6], errors="coerce")

contagens = (
    amostra.dropna(subset=["ano_epi", "semana_epi"])
    .astype({"ano_epi": int, "semana_epi": int})
    .groupby(["doenca", "ano_epi", "semana_epi"])
    .size()
    .rename("notificacoes")
    .reset_index()
)
serie = (
    calendario(min(ANOS_AMOSTRA), max(ANOS_AMOSTRA))
    .merge(pd.DataFrame({"doenca": DOENCAS_SINAN}), how="cross")
    .merge(contagens, how="left", on=["doenca", "ano_epi", "semana_epi"])
    .fillna({"notificacoes": 0})
    .sort_values(["doenca", "inicio"])
)

px.line(
    serie,
    x="inicio",
    y="notificacoes",
    facet_row="doenca",
    height=650,
    title="Notificações por semana de início dos sintomas",
    labels={"inicio": "", "notificacoes": "notificações"},
).update_yaxes(matches=None)

In [ ]:
anuais = serie.groupby(["doenca", "ano_epi"])["notificacoes"].sum().unstack(0)
display(anuais.astype(int))
px.bar(
    anuais,
    barmode="group",
    title="Notificações por ano epidemiológico",
    labels={"value": "notificações"},
)

In [ ]:
perfil_sazonal = (
    serie.groupby(["doenca", "semana_epi"])["notificacoes"]
    .quantile([0.25, 0.5, 0.75])
    .unstack()
    .reset_index()
    .rename(columns={0.25: "p25", 0.5: "mediana", 0.75: "p75"})
)
px.line(
    perfil_sazonal,
    x="semana_epi",
    y=["p25", "mediana", "p75"],
    facet_row="doenca",
    height=650,
    title="Perfil sazonal: quartis das notificações por semana epidemiológica",
).update_yaxes(matches=None)

In [ ]:
def estatisticas(g: pd.DataFrame) -> pd.Series:
    y = g["notificacoes"]
    media_sazonal = g.groupby("semana_epi")["notificacoes"].transform("mean")
    validos = media_sazonal > 0
    # Dispersão de Pearson em torno da média de cada semana do ano: ~1 sob Poisson.
    pearson = ((y - media_sazonal)[validos] ** 2 / media_sazonal[validos]).sum() / (
        validos.sum() - g["semana_epi"].nunique()
    )
    fev_mai = g["semana_epi"].between(6, 22)
    return pd.Series(
        {
            "semanas": len(y),
            "média semanal": round(y.mean(), 1),
            "variância / média": round(y.var() / y.mean(), 1),
            "dispersão após sazonalidade": round(pearson, 1),
            "% semanas com zero": round(100 * (y == 0).mean(), 1),
            "% notificações nas SE 6 a 22 (fev-mai)": round(100 * y[fev_mai].sum() / y.sum(), 1),
            "semana de pico da mediana": int(
                g.groupby("semana_epi")["notificacoes"].median().idxmax()
            ),
            "maior ano / mediana anual": round(
                g.groupby("ano_epi")["notificacoes"].sum().pipe(lambda s: s.max() / s.median()), 1
            ),
            "autocorrelação lag 1": round(y.autocorr(1), 2),
            "autocorrelação lag 4": round(y.autocorr(4), 2),
            "autocorrelação lag 52": round(y.autocorr(52), 2),
        }
    )


estatisticas_serie = serie.groupby("doenca")[serie.columns].apply(estatisticas).T
estatisticas_serie

#### Número a bater

Erro das previsões ingênuas nas últimas 104 semanas, um passo à frente. O MASE usa como
escala o erro do baseline sazonal no período de treino; abaixo de 1, o modelo supera o
baseline sazonal.

In [ ]:
def comparar_baselines(g: pd.DataFrame, semanas_teste: int = 104) -> pd.DataFrame:
    y = g["notificacoes"].reset_index(drop=True)
    corte = len(y) - semanas_teste
    teste, treino = y[corte:], y[:corte]
    previsoes = {"sazonal (t-52)": baseline_sazonal(y), "ingênua (t-1)": y.shift(1)}
    return pd.DataFrame(
        {
            nome: {"MAE": mae(teste, p[corte:]), "MASE": mase(teste, p[corte:], treino)}
            for nome, p in previsoes.items()
        }
    ).T.round(2)


pd.concat({d: comparar_baselines(g) for d, g in serie.groupby("doenca")})

### 2.6 Febre amarela (casos humanos)

In [ ]:
febre_amarela = pd.DataFrame(await cliente.todos(ENDPOINTS["febre_amarela"].caminho))
print(
    f"{len(febre_amarela)} registros, {febre_amarela.duplicated().sum()} linhas idênticas repetidas"
)
anos_fa = febre_amarela["ano_is"]
print(f"anos de início dos sintomas: {anos_fa.min()} a {anos_fa.max()}")
print(f"letalidade (obito == 'SIM'): {100 * (febre_amarela['obito'] == 'SIM').mean():.1f}%")

por_ano = febre_amarela.groupby("ano_is").size()
display(
    px.bar(
        por_ano, title="Casos humanos de febre amarela por ano", labels={"value": "casos"}
    ).update_layout(showlegend=False)
)
febre_amarela.groupby("uf_lpi").size().nlargest(10).rename("casos").to_frame().T

## Parte 3: síntese

In [ ]:
sintese = pd.Series(
    {
        "Autenticação": verificacao_api.loc["Autenticação", "observado"],
        "Registros por requisição": verificacao_api.loc["Tamanho máximo da página", "observado"],
        "offset": verificacao_api.loc["Semântica do offset", "observado"],
        "UBS no cadastro": qualidade_ubs["registros"],
        "UBS sem coordenada válida": qualidade_ubs["coordenada ausente ou ilegível"]
        + qualidade_ubs["coordenada fora do Brasil"],
        "Linhas de UBS com CNES repetido": qualidade_ubs["linhas com CNES repetido"],
        "Municípios sem UBS": qualidade_ubs["municípios sem nenhuma UBS"],
        "Notificações na amostra": len(amostra),
        "Dispersão da dengue após sazonalidade": estatisticas_serie.loc[
            "dispersão após sazonalidade", "dengue"
        ],
    }
)
sintese.to_frame("valor")

### Achados da execução de 2026-10-28

Os números abaixo são desta execução; as tabelas acima são recalculadas a cada rodada.

**API DEMAS**

- Sem autenticação; `limit` até 1000; `offset` por registro; `id_municip` só com 6 dígitos.
- Sem o parâmetro `campos`, a amostra inteira (17.314 notificações, 44 requisições) baixou
  em 4 s; o cadastro completo de UBS (47 requisições), também em 4 s.

**Esquema**

- Dengue tem 196 colunas, chikungunya 125 e zika 41, com 39 em comum. Na primeira página
  de dengue de 2024, 79 colunas vieram 100% vazias.
- Os tipos mudam entre agravos: `id_municip` chega como inteiro na chikungunya e
  `nu_idade_n` como float. A ausência aparece como `None`, `NaN` ou a string `"nan"`.
- Os códigos de `classi_fin` dependem do sistema de origem. Registros do SINAN NET
  (`tp_sistema` 1) usam 1 (confirmado) e 2 (descartado); os do SINAN Online usam 5, 8 e
  10 a 13. A chikungunya de 2015 e 2016 mistura os dois, e `tp_sistema` vem vazio em
  parte dos registros (toda a dengue de 2015 e 2016). O mapeamento tem de ser feito pelo
  código, não pelo sistema.
- Febre amarela: datas em `dd/mm/aaaa`, códigos numéricos como float e 32 linhas idênticas
  repetidas em 2.913.

**UBS**

- 46.848 unidades. 2.166 (4,6%) sem coordenada legível, 3 fora do Brasil e 825 com até
  duas casas decimais na latitude (precisão de cerca de 1 km).
- Nenhum CNES repetido, 1 código IBGE fora do dicionário e 91 municípios sem UBS no
  cadastro.

**Notificações da amostra (Olinda, 2015 a 2025)**

- 27% das notificações de dengue feitas em Olinda são de residentes de outros municípios
  (21,5% na chikungunya). Contar por residência exige baixar o estado inteiro, porque a
  API só filtra pelo município notificador.
- Atraso de notificação: mediana de 1 a 4 dias; p90 de 6 dias na dengue e de 35 na zika.
  As semanas mais recentes ficam incompletas no momento da extração.
- Na dengue, 41% das notificações foram descartadas e 18% ficaram inconclusivas, então
  notificações totais e casos prováveis divergem bastante.
- Nenhuma notificação de zika em Olinda em 2015 e 2016, apesar da epidemia em Pernambuco.
  É uma lacuna de cobertura a investigar.

**Série temporal**

- Sobredispersão forte: variância sobre média de 69 na dengue e 59 na chikungunya. Mesmo
  em torno da média de cada semana do ano, a dispersão fica em 50 e 43, contra 1 sob
  Poisson. A binomial negativa é a verossimilhança adequada.
- A sazonalidade é regional. Em Olinda, o pico mediano da dengue cai na semana 29 (julho)
  e só 49% das notificações caem entre fevereiro e maio. O modelo precisa aprender a
  sazonalidade por região ou município, em vez de assumir o pico de fevereiro a maio.
- Anos epidêmicos: 2015 na dengue (6,4 vezes a mediana anual) e 2016 na chikungunya
  (23 vezes).
- Autocorrelação de 0,92 no lag 1 e de 0,18 no lag 52. Nas últimas 104 semanas, a
  previsão ingênua t-1 teve MAE de 7,8 na dengue, contra 23,3 do baseline sazonal. Para
  horizonte de uma semana, o número a bater é o t-1.
- O MASE depende de quais anos entram na escala: na chikungunya, o treino inclui 2016 e o
  MASE cai para 0,10. Para comparar municípios, use uma janela de treino comum.

### Decisões propostas para o ETL e o modelo (a validar)

1. Agregar por município de residência e semana de início dos sintomas, como nos boletins
   do MS. Isso pede recorte inicial por UF inteira, ou a carga pelos CSVs do OpenDataSUS.
2. Mapear `classi_fin` pelo código: descartado {2, 5}, confirmado {1, 10, 11, 12, 13},
   inconclusivo {8}. Publicar notificações totais e casos prováveis.
3. Normalizar ausências e tipos na extração; códigos sempre como texto.
4. Sinalizar as semanas recentes como incompletas no dashboard e na previsão.
5. UBS: descartar as unidades sem coordenada válida e marcar as de baixa precisão.
6. Febre amarela: remover duplicatas e tratar como série à parte, com o ano de
   monitoramento de julho a junho.